# Procesamiento de lenguaje natural (NLP)

---
Curso sabatino

Por: Cesar Macias

Duración: 5 horas

---

Contenido:

1. El lenguaje natural
2. Conocimiento del lenguaje
3. Vectorización
4. Extracción de entidades (NER)
5. Análisis de sentimientos
6. API de Open AI

---

## ¿Qué es el análisis de sentimientos?

El análisis de sentimientos es una técnica del PLN usada para minar la opinión de las personas respecto a agún producto o algo en específico. Por ejemplo en esta sección utilizaremos el conjunto de datos de IMDB que contiene reseñas de películas y el sentimiento que expresan quienes opinan de las mismas en general. Las posibles categorías son:
- 0: negativo
- 1: positivo

### Implementemos un análisis de sentimientos con modelos preentrenados

#### 1. Instalamos las librerías necesarias y hacemos las importaciones

In [ ]:
!pip install transformers==4.49.0 datasets==3.6.0

In [ ]:
!git clone https://github.com/CCogS-Mx/text-preprocessing.git
!git clone https://github.com/CCogS-Mx/fine-tuning-bert-and-friends.git
import sys
sys.path.append('./text-preprocessing/scripts/')
sys.path.append('./fine-tuning-bert-and-friends/scripts/')

fatal: destination path 'text-preprocessing' already exists and is not an empty directory.
fatal: destination path 'fine-tuning-bert-and-friends' already exists and is not an empty directory.


In [ ]:
import text_preprocessing as tp
from fine_tuning_BERT import BertFineTunningMultiClassText
from datasets import load_dataset

#### 2. Preparamos los datos

In [ ]:
corpus = load_dataset('imdb')
corpus

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

Podemos ver que ya existen las particiones de entrenamiento, prueba y validación

In [ ]:
import pandas as pd
train_data = pd.DataFrame({'text':corpus['train'][:]['text'],'label':corpus['train'][:]['label']})
test_data = pd.DataFrame({'text':corpus['test'][:]['text'],'label':corpus['test'][:]['label']})

In [ ]:
train_data

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [ ]:
test_data

,text,label
0,I love sci-fi and am willing to put up with a ...,0
1,"Worth the entertainment value of a rental, esp...",0
2,its a totally average film with a few semi-alr...,0
3,STAR RATING: ***** Saturday Night **** Friday ...,0
4,"First off let me say, If you haven't enjoyed a...",0
...,...,...
24995,Just got around to seeing Monster Man yesterda...,1
24996,I got this as part of a competition prize. I w...,1
24997,I got Monster Man in a box set of three films ...,1
24998,"Five minutes in, i started to feel how naff th...",1


In [3]:
## Ejercicio, preprocesa los datos antes de hacer el fine tunning
train_data['text'][0] = train_data['text'].apply(tp.pre_process_text)
test_data['text'] = test_data['text'].apply(tp.pre_process_text)

NameError: name 'train_data' is not defined

In [ ]:
x_train = list(train_data['text'])
y_train = list(train_data['label'])

x_test = list(test_data['text'])
y_test = list(test_data['label'])

In [ ]:
import os
#os.mkdir("model")

In [ ]:
model = BertFineTunningMultiClassText(X_train= x_train,
                                           y_train= y_train,
                                           pretrained_model= 'bert-base-uncased',
                                           epochs= 3,
                                           n_labels = 2,
                                           batch_size= 8,
                                           average= 'macro',
                                           learning_rate = 2e-5,
                                           random_state= 42,
                                           val_size = 0.2,
                                           eps= 1e-8,
                                           warmup_steps= 0,
                                           saving_path = '/content/model/')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
model.fine_tune()

  0%|          | 0/3 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/2500 [00:00<?, ?it/s]


Epoch1
Validation loss: 0.19523249602280557
Accuracy: 0.9216
Precision (binary): 0.898036253776435
Recall (binary): 0.9512
F1-score (binary): 0.9238539238539238


Epoch 2:   0%|          | 0/2500 [00:00<?, ?it/s]


Epoch2
Validation loss: 0.1882100490545854
Accuracy: 0.9348
Precision (binary): 0.9480626545754328
Recall (binary): 0.92
F1-score (binary): 0.9338205440519691


Epoch 3:   0%|          | 0/2500 [00:00<?, ?it/s]


Epoch3
Validation loss: 0.25006175065431746
Accuracy: 0.935
Precision (binary): 0.9469790382244143
Recall (binary): 0.9216
F1-score (binary): 0.9341171700790594


In [ ]:
import torch
preds = model.test(x_test, y_test, path_to_state_dict= '/content/model/')
torch.cuda.empty_cache()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Accuracy: 0.93296
Precision (binary): 0.9441891004596191
Recall (binary): 0.92032
F1-score (binary): 0.9321017663263652


In [ ]:
y_pred = []
for pred in preds:
  y_pred.append(pred.argmax())

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, digits= 4))

              precision    recall  f1-score   support

           0     0.9223    0.9456    0.9338     12500
           1     0.9442    0.9203    0.9321     12500

    accuracy                         0.9330     25000
   macro avg     0.9332    0.9330    0.9329     25000
weighted avg     0.9332    0.9330    0.9329     25000

